# 11 — Python Interpreter and Bytecode

## Learning objectives

By the end of this notebook, you should be able to:

- Explain the high-level steps between Python source code and program execution.
- Describe the roles of the interpreter, compiler stage, bytecode, and Python Virtual Machine (PVM).
- Inspect bytecode using Python's `dis` module.
- Understand what `.pyc` files and the `__pycache__` directory are for.
- Distinguish source code, bytecode, and runtime execution.
- Recognize that implementation details can vary between Python implementations and versions.

> **Scope:** This notebook uses CPython as the main example because it is the standard Python implementation most learners use. Exact internals can differ in other Python implementations.

## 1. The big picture

When you write Python code, the computer does not directly execute the characters in your `.py` file as machine instructions. In CPython, a simplified execution path looks like this:

```text
Python source code (.py)
          |
          v
   Parse and compile
          |
          v
     Python bytecode
          |
          v
 Python Virtual Machine
          |
          v
     Program results
```

This diagram is a high-level model. Real execution involves additional details, such as parsing, code objects, runtime object operations, and implementation-specific optimizations.

The main idea is that CPython compiles source code into bytecode and then executes that bytecode in its runtime.

## 2. Source code: the code you write

**Source code** is the human-readable Python code stored in a file or entered interactively.

For example, a file named `hello.py` might contain:

```python
message = "Hello, Python!"
print(message)
```

The `.py` file is source code. It is designed to be read and edited by programmers.

In this notebook, code cells are also source code. When you run a cell, the notebook's Python kernel processes the code using the active Python implementation.

In [1]:
message = "Hello, Python!"
print(message)

Hello, Python!


**Expected output**

```text
Hello, Python!
```

This simple example will help us connect source code to the execution model.

## 3. What does the interpreter do?

The **Python interpreter** is the software that processes Python code and runs it. In CPython, several stages are involved:

1. **Parsing:** Python checks whether the source follows Python's grammar and builds an internal representation.
2. **Compilation:** Python compiles the parsed code into a code object containing bytecode and related information.
3. **Execution:** The CPython runtime executes the bytecode and performs operations on Python objects.

These stages are connected, but they are not the same thing. For example, a syntax error can be detected before the program begins its ordinary execution.

## 4. Compilation does not necessarily mean machine code

The word **compilation** can mean different things in different language implementations.

For CPython, ordinary Python source code is compiled into **Python bytecode**. This bytecode is a lower-level, implementation-specific instruction format; it is not generally the same as native machine code for your CPU.

A simplified comparison:

| Stage | Meaning |
|---|---|
| Source code | The Python instructions you write |
| Bytecode | Lower-level instructions for the Python runtime |
| Runtime execution | The interpreter carries out those instructions and interacts with Python objects |
| Native machine code | CPU instructions executed directly by a processor |

Some Python implementations use different execution strategies or additional compilation techniques. For this notebook, focus on the familiar CPython model.

## 5. What is bytecode?

**Bytecode** is a sequence of instructions generated by CPython's compiler. The Python Virtual Machine executes these instructions.

Consider:

```python
total = 10 + 20
print(total)
```

The source code is readable Python. Internally, CPython compiles it into instructions that represent operations such as loading constants, performing an addition, assigning a name, and calling a function.

The exact bytecode instructions can change between Python versions. Learn the concept, not a fixed list of opcodes.

In [2]:
import dis

def add_numbers():
    total = 10 + 20
    print(total)

dis.dis(add_numbers)

  3           RESUME                   0

  4           LOAD_SMALL_INT          30
              STORE_FAST               0 (total)

  5           LOAD_GLOBAL              1 (print + NULL)
              LOAD_FAST_BORROW         0 (total)
              CALL                     1
              POP_TOP
              LOAD_CONST               1 (None)
              RETURN_VALUE


### Understanding the `dis` example

The `dis` module is a standard-library tool for disassembling Python code objects. It displays bytecode instructions in a human-readable form.

When you run the previous cell, you should see a list of instructions with offsets and operation names. The exact output depends on your Python version.

Notice that `dis.dis()` is inspecting the compiled instructions associated with the function. It is not printing the source code again, and the output is not CPU machine code.

## 6. What is the Python Virtual Machine (PVM)?

The **Python Virtual Machine (PVM)** is a useful name for the runtime mechanism that executes Python bytecode.

A simplified analogy:

- Source code is a recipe written for people.
- Bytecode is a more structured set of instructions.
- The PVM is the runtime that follows those instructions.

This is only an analogy. The PVM is not a separate physical computer; in CPython, bytecode execution is implemented by the CPython runtime running as a native program on your operating system.

The interpreter and PVM are closely related parts of the overall Python implementation, rather than completely independent applications you need to install separately.

## 7. A small experiment with bytecode

Let's inspect a simple function. The `dis` module lets us explore the compiled instructions without needing to understand every opcode.

```python
def square(number):
    return number * number
```

The function's source is straightforward. Disassembling it reveals the lower-level instructions CPython uses to implement its behavior.

In [3]:
import dis

def square(number):
    return number * number

dis.dis(square)

  3           RESUME                   0

  4           LOAD_FAST_BORROW_LOAD_FAST_BORROW 0 (number, number)
              BINARY_OP                5 (*)
              RETURN_VALUE


**How to read the result**

Your output will contain instructions and offsets. Depending on the Python version, you may see names such as `LOAD_FAST`, `BINARY_OP`, and `RETURN_VALUE`, or different instructions in a future version.

At a conceptual level:

- Python loads the function's input.
- It performs the multiplication.
- It returns the result.

Do not memorize the exact opcode names. The purpose is to see that source code is compiled into a sequence of runtime instructions.

In [4]:
print(square(5))
print(square(8))

25
64


**Expected output**

```text
25
64
```

This cell calls the function. The previous `dis.dis()` cells inspect bytecode; this cell demonstrates the function's behavior.

## 8. What is a `.pyc` file?

A `.pyc` file stores cached compiled bytecode for a Python module. CPython can use cached bytecode to avoid recompiling a module when the cache is valid.

When Python imports a module from source, it may create a cache file in a directory named `__pycache__`.

A typical project might look like this:

```text
my_project/
├── main.py
├── helpers.py
└── __pycache__/
    └── helpers.cpython-3xx.pyc
```

The exact filename includes implementation and version details and can vary. The cache directory may not appear in every situation, and Python can run without a visible `.pyc` file.

## 9. What is `__pycache__`?

`__pycache__` is a directory where CPython commonly stores cached bytecode files for imported modules.

Important details:

- `.py` files contain source code.
- `.pyc` files contain cached compiled bytecode.
- `__pycache__` is a directory, not a Python keyword.
- Cache files are an implementation detail and can usually be regenerated.
- Deleting a cache file generally does not delete your source code.
- Python validates cache information before using a cached file; it does not blindly trust every `.pyc` file.

You generally should not edit `.pyc` files manually.

## 10. Demonstrate bytecode caching with a small project

The next code cell creates a temporary directory, writes a small module, imports it, and checks whether Python created a cache directory. It uses only the standard library.

The example deliberately uses a temporary directory so it does not clutter your learning repository. Cache creation can depend on the Python implementation and execution environment, so the code reports what it observes rather than assuming a cache file must exist.

In [5]:
from pathlib import Path
from tempfile import TemporaryDirectory
import importlib
import sys

with TemporaryDirectory() as temp_dir:
    project_dir = Path(temp_dir)
    module_path = project_dir / "greeting_helper.py"
    module_path.write_text(
        'def greet(name):\n'
        '    return f"Hello, {name}!"\n',
        encoding="utf-8",
    )

    sys.path.insert(0, str(project_dir))
    try:
        importlib.invalidate_caches()
        module = importlib.import_module("greeting_helper")

        print(module.greet("Python"))
        cache_dir = project_dir / "__pycache__"
        print("Cache directory exists:", cache_dir.exists())

        if cache_dir.exists():
            print("Cache files:", [path.name for path in cache_dir.iterdir()])
    finally:
        sys.path.remove(str(project_dir))
        sys.modules.pop("greeting_helper", None)

Hello, Python!
Cache directory exists: True
Cache files: ['greeting_helper.cpython-314.pyc']


**Expected behavior**

The greeting should be:

```text
Hello, Python!
```

On CPython, a cache directory and `.pyc` file will commonly be created when importing this source module. The code prints whether the directory exists and lists any files found. Other implementations or special execution environments may behave differently.

The temporary directory is cleaned up automatically when the `with` block ends.

## 11. Does every Python execution create a `.pyc` file?

No. This is an important distinction.

- Running a script directly does not mean that a `.pyc` cache for that script must be created.
- Importing a source module commonly causes CPython to write cached bytecode, when permitted.
- Cache writing can be disabled or unavailable.
- Interactive commands and notebook cells do not necessarily create a normal module `.pyc` file.

The `.pyc` cache is primarily an import optimization. It is not required for Python programs to work.

## 12. Syntax errors versus runtime errors

Because Python processes source code before executing it, some mistakes are detected during parsing or compilation.

### Syntax error

```python
# Uncomment the invalid line to test it:
# if True print("Hello")
```

The example line is invalid because the `if` statement is missing a colon. Python reports a `SyntaxError` when it processes that source.

### Runtime error

```python
print(10 / 0)
```

This is syntactically valid, but raises `ZeroDivisionError` when executed.

These are different kinds of errors:

- A **syntax error** means the source cannot be processed as valid Python syntax.
- A **runtime error** occurs while valid code is being executed.

For a clean notebook, the invalid example above remains commented out, so the cell does not fail when run.

In [6]:
# Valid syntax, but the operation would fail if executed:
# print(10 / 0)

print("This cell contains valid Python syntax.")

This cell contains valid Python syntax.


**Expected output**

```text
This cell contains valid Python syntax.
```

## 13. How can you inspect bytecode yourself?

You can use `dis` on a function or a code object:

```python
import dis

def multiply(a, b):
    return a * b

dis.dis(multiply)
```

You can also inspect the bytecode bytes on a code object using its `co_code` attribute:

```python
print(multiply.__code__.co_code)
```

This displays a bytes object, not a friendly explanation of each instruction. Prefer `dis.dis()` when learning because it formats the instructions for inspection.

Exact bytecode is version-specific. If an online example shows different instructions from your output, check which Python version it uses.

In [7]:
import dis

def multiply(a, b):
    return a * b

print("Bytecode bytes:", multiply.__code__.co_code)
print("Disassembled instructions:")
dis.dis(multiply)

Bytecode bytes: b'\x80\x00W\x01,\x05\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00#\x00'
Disassembled instructions:
  3           RESUME                   0

  4           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP                5 (*)
              RETURN_VALUE


## 14. Common misconceptions

1. **“Python directly executes the text in a `.py` file as CPU instructions.”**  
   In CPython, source code is compiled into bytecode and executed by the runtime.

2. **“Bytecode is the same as machine code.”**  
   CPython bytecode is an instruction format for the Python runtime, not generally native CPU instructions.

3. **“The PVM is a separate computer.”**  
   It is a conceptual name for the bytecode execution mechanism within the implementation.

4. **“Every script creates a `.pyc` file.”**  
   No. Bytecode caching is especially associated with importing modules.

5. **“A `.pyc` file replaces the source file.”**  
   Usually it is a cache alongside the source; the `.py` file remains the editable source.

6. **“Bytecode instructions never change.”**  
   They can change between Python versions and implementations.

## 15. Practice — check your understanding

### Questions

1. What is Python source code?
2. In CPython, what is bytecode?
3. What does the Python Virtual Machine do?
4. What is the purpose of the `dis` module?
5. What does a `.pyc` file store?
6. Where are cached bytecode files commonly stored?
7. Does every script execution create a `.pyc` file?
8. What is the difference between a syntax error and a runtime error?
9. Why might bytecode output differ between two Python versions?
10. Why should you avoid editing `.pyc` files manually?

### Suggested answers

1. The readable Python instructions written by a programmer.
2. Compiled instructions executed by the CPython runtime.
3. It executes Python bytecode.
4. It disassembles code objects so their bytecode instructions can be inspected.
5. Cached compiled bytecode.
6. In `__pycache__`, commonly for imported source modules.
7. No; cache creation depends on how the code is run and the environment.
8. A syntax error prevents the source from being processed as valid syntax; a runtime error happens during execution.
9. Bytecode instructions and compiler behavior can change between versions.
10. They are generated cache files and are not intended to be hand-edited.

## 16. Summary

- CPython parses and compiles Python source code into code objects containing bytecode.
- The CPython runtime executes bytecode and interacts with Python objects.
- The `dis` module helps inspect bytecode instructions.
- `.pyc` files store cached bytecode, commonly for imported modules.
- `__pycache__` is the usual cache directory.
- Bytecode is not the same as native machine code.
- Exact implementation details can change across Python versions.
- Syntax errors and runtime errors occur at different stages.

**Next notebook:** `12_Python_Program_Structure.ipynb`

There, we will learn how to organize a Python program, use imports, understand top-level code, and work with the `if __name__ == "__main__":` entry-point pattern.